# 강의 04 · 실습 3 — 서브그래프 모듈화 · (4) 고난도 I

## 1. 문제상황

- 온라인 서점 고객센터는 메일 창구와 실시간 채팅 창구를 따로 운영합니다.
- 두 창구의 문의는 같은 기준(배송·파손·환불)으로 분류되어야 하는데, 지금은 메일 팀과 채팅 팀이 분류 코드를 각자 가지고 있습니다.
- 분류 기준이 바뀌면 두 팀이 따로 고치고, 고치는 시점이 어긋나서 같은 문의가 창구에 따라 다르게 분류되는 일이 생깁니다.
- 메일 그래프의 상태 키는 email·category이고 채팅 그래프의 상태 키는 message·kind라서, 분류 코드를 한쪽에서 가져와도 그대로 붙지 않습니다.
- 채팅 창구는 분류 결과에 따라 첫 응답 문장까지 자동으로 보내야 하므로, 메일 그래프에는 없는 노드가 하나 더 필요합니다.

## 2. 문제와 목표

- **문제**: 같은 분류 절차가 두 그래프에 따로 적혀 있어 기준이 어긋나고, 두 그래프의 상태 키가 서로 달라 분류 코드를 그대로 옮길 수 없습니다.
- **목표**
  - 분류 절차를 자식 그래프 하나(text → label·urgent)로 컴파일합니다.
  - 메일 부모와 채팅 부모가 각자의 래퍼(wrapper) 함수로 같은 자식을 재사용하게 만듭니다.
    - 부모 둘: 메일 부모(상태 키 email·category·priority·handled), 채팅 부모(상태 키 message·kind·answer)
  - 채팅 부모에는 분류 결과에 따라 첫 응답 문장을 만드는 노드를 하나 더 둡니다.
- **목표 달성 여부의 판정 기준**:
  - 같은 내용의 찢어진 책 문의를 메일 부모와 채팅 부모에 넣었을 때,
  - 두 부모의 분류 결과가 「파손」으로 같고, 메일 부모는 배정 메시지를, 채팅 부모는 첫 응답 문장을 만들며,
  - 자식 그래프는 한 번만 컴파일되고 두 부모의 래퍼 함수가 그 하나를 부르는 것을 코드 구성에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex03_s4_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 자식 전용 상태 `SubState`는 분류할 글(`text`), 분류 결과(`label`), 긴급 여부(`urgent`, 참/거짓) 키 세 개입니다.
    - 메일 부모 상태 `MailState`는 `email`·`category`·`priority`·`handled` 키 네 개, 채팅 부모 상태 `ChatState`는 채팅 문장(`message`), 분류 결과(`kind`), 첫 응답(`answer`) 키 세 개입니다.
2. **자식 노드 함수를 만듭니다.**
    - classify 노드는 자식 상태의 `text`를 읽어 배송·파손·환불 중 한 단어를 `label` 키에 씁니다.
    - judge_urgent 노드는 분류 결과가 파손이거나 본문에 「급」이 들어 있으면 `urgent` 키에 `True`를, 아니면 `False`를 씁니다.
    - 모델을 부르지 않는 규칙 노드입니다.
3. **부모 노드 함수를 만듭니다.**
    - handle 노드는 메일 부모 상태의 category와 priority로 「<분류> 담당자에게 <긴급도> 배정」을 `handled` 키에 씁니다.
    - first_reply 노드는 채팅 부모 상태의 kind를 읽어, 파손이면 「사진을 보내 주시면 바로 교환해 드립니다.」, 환불이면 「주문번호를 알려 주시면 환불을 접수합니다.」, 그 밖에는 「배송 상태를 확인해 안내해 드립니다.」를 `answer` 키에 씁니다.
4. **자식 그래프를 구성하고 컴파일합니다.**
    - 자식 상태로 classify → judge_urgent를 이어 한 번만 컴파일합니다.
5. **래퍼 함수 두 개를 만들고 두 부모 그래프에 노드를 등록합니다.**
    - 메일용 래퍼 함수는 email → text로 넣고 label → category, urgent → priority(긴급/일반)로 되받습니다.
    - 채팅용 래퍼 함수는 message → text로 넣고 label → kind로만 되받습니다.
    - 메일 부모에는 classifier·handle을, 채팅 부모에는 classifier·first_reply를 등록합니다.
6. **엣지를 연결합니다.**
    - 메일 부모는 START → classifier → handle → END, 채팅 부모는 START → classifier → first_reply → END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 같은 내용의 찢어진 책 문의를 메일 부모와 채팅 부모에 각각 넣고, 노드가 하나 끝날 때마다 바뀐 키를 출력합니다.
    - 자식 그래프의 컴파일은 단계 ③-a의 한 번뿐이며, 두 래퍼 함수는 그 객체 하나를 부릅니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 자식 그래프는 한 번 세워 두 부모가 재사용하며, 래퍼 함수는 부모마다 따로 만듭니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 자식 상태와 부모 상태 두 개의 키를 선언합니다 | `class SubState`, `class MailState`, `class ChatState` | 1 |
| ② 노드 함수 정의 | 자식 노드와 부모 노드 함수를 만듭니다 | `def classify(state) -> dict`, `def first_reply(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 자식을 한 번 컴파일하고, 부모마다 래퍼 함수를 만들어 등록합니다 | `CHILD = child.compile()`, `add_node("classifier", call_for_mail)` | 4, 5 |
| ④ 엣지 연결 | 두 부모의 순서를 정합니다 | `add_edge` | 6 |
| ⑤ 컴파일과 실행 | 두 부모를 컴파일하고 같은 문의를 넣어 실행합니다 | `compile()`, `stream()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")


# 주어진 자료
TEXT = "주문한 책이 찢어진 채로 왔습니다. 급하게 교환하고 싶어요."   # 메일 부모와 채팅 부모에 같은 문면으로 넣는다


### 단계 ① — 상태 정의 (요구사항 1)

In [ ]:
# 여기에 단계 ①(자식 상태와 부모 상태 두 개 정의)을(를) 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

In [ ]:
# 여기에 단계 ②(자식 노드 두 개, 부모 노드 두 개 정의)을(를) 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4, 5)

#### 단계 ③-a — 자식 그래프 구성과 컴파일 (요구사항 4)

In [ ]:
# 여기에 단계 ③-a(자식 그래프 구성과 컴파일)을(를) 작성합니다.

#### 단계 ③-b — 래퍼 함수 두 개 정의와 두 부모 그래프 노드 등록 (요구사항 5)

자식은 하나이고 래퍼 함수는 부모마다 다릅니다. 부모의 키 이름이 다르면 번역만 달라지고 자식은 그대로입니다.

In [ ]:
# 여기에 단계 ③-b(래퍼 함수 두 개 정의, 두 부모 빌더 생성과 노드 등록)을(를) 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

In [ ]:
# 여기에 단계 ④(두 부모의 엣지 연결)을(를) 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`graph.stream(입력, stream_mode="updates")`로 부르면 노드가 하나 끝날 때마다 `{노드 이름: 바뀐 키}`가 나옵니다.


In [ ]:
# 여기에 단계 ⑤(두 부모의 컴파일과 실행, 자식 객체 동일성 확인)을(를) 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. 메일 부모와 채팅 부모의 분류 결과가 「파손」으로 같습니다. 분류 코드는 자식 그래프 한 곳에만 있습니다.
2. 메일 부모의 `[classifier]`는 `category`와 `priority` 키 두 개를, 채팅 부모의 `[classifier]`는 `kind` 키 하나를 돌려줍니다. 같은 자식이 돌려준 값을 래퍼 함수가 부모마다 다른 키 이름으로 되돌렸습니다.
3. 메일 부모는 `[handle]`이 배정 메시지를, 채팅 부모는 `[first_reply]`가 첫 응답 문장을 만듭니다. 부모 노드는 서로 다르고 자식은 하나입니다.